In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt
from itertools import combinations
from statsmodels.tsa.stattools import coint


In [2]:
UNIVERSE = [
    "BTC",
    "ETH",
    "SOL",
    "BNB",
    "XRP",
    "DOGE",
    "ADA",
    "LINK",
    "AVAX",
    "DOT",
    "LTC",
    "ATOM",
    "AAVE",
    "UNI",
    "ARB",
    "OP",
    "SUI",
    "NEAR",
    "ALGO",
    "FIL"
]

In [3]:
SYMBOLS = [f"{coin}/USD" for coin in UNIVERSE]

In [4]:
exchange = ccxt.kraken()
markets = exchange.load_markets()

symbols = []

for coin in UNIVERSE:
    pair = f"{coin}/USD"
    if pair in markets:
        symbols.append(pair)

print(symbols)

['BTC/USD', 'ETH/USD', 'SOL/USD', 'BNB/USD', 'XRP/USD', 'DOGE/USD', 'ADA/USD', 'LINK/USD', 'AVAX/USD', 'DOT/USD', 'LTC/USD', 'ATOM/USD', 'AAVE/USD', 'UNI/USD', 'ARB/USD', 'OP/USD', 'SUI/USD', 'NEAR/USD', 'ALGO/USD', 'FIL/USD']


In [5]:
pairs = list(combinations(symbols, 2))

In [6]:
len(set(pairs))

190

In [7]:
def ohlcv_to_df(data, symbol):
    df = pd.DataFrame(data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[["timestamp", "close"]].copy()
    df = df.rename(columns={"close": symbol})
    return df

In [8]:
def test_pairs(pair, window):
    A_raw = pair[0]
    B_raw = pair[1]

    A = exchange.fetch_ohlcv(A_raw, timeframe="1h", limit=1000)
    B = exchange.fetch_ohlcv(B_raw, timeframe="1h", limit=1000)

    A_name = A_raw.split("/")[0]
    B_name = B_raw.split("/")[0]

    A = ohlcv_to_df(A, A_name)
    B = ohlcv_to_df(B, B_name)
    
    log_price_A = np.log(A[A_name]).diff().dropna()
    log_price_B = np.log(B[B_name]).diff().dropna()

    cor = log_price_A.corr(log_price_B)

    score, pvalue, _ = coint(log_price_A, log_price_B)

    X = sm.add_constant(log_price_A)
    y = log_price_B

    ols = sm.OLS(y, X).fit()
    # print(ols.summary())

    beta = ols.params[ols.params.index[1]]
    spread = ols.resid
    spread_vol = spread.std()

    spread_lag = spread.shift(1)
    spread_diff = spread - spread_lag
    df = pd.concat([spread_diff, spread_lag], axis=1).dropna()
    df.columns = ["diff", "lag"]
    model = sm.OLS(df["diff"], sm.add_constant(df["lag"])).fit()
    lam = model.params["lag"]
    half_life = -np.log(2) / lam

    z = (spread - spread.mean()) / spread.std()

    upper = ((z.shift(1) < 2) &(z >= 2)).sum()
    lower = ((z.shift(1) > -2) & (z <= -2)).sum()
    crossings = upper + lower

    df = pd.DataFrame({
        "Pair": f"{A_name}/{B_name}",
        "Corr": cor,
        "Score": score,
        "P-value": pvalue,
        "Beta": beta,
        "Spread Vol": spread_vol,
        "Half-life": half_life,
        "Crossings": crossings
    }, index=[0])
    
    return df
    

In [9]:
df_list = []

In [10]:
for pair in pairs:
    df = test_pairs(pair, 25)
    df_list.append(df)

In [11]:
final_df = pd.concat(df_list, ignore_index=True)

In [12]:
final_df.shape

(190, 8)

In [13]:
final_df.drop_duplicates().shape

(190, 8)

In [14]:
final_df

,Pair,Corr,Score,P-value,Beta,Spread Vol,Half-life,Crossings
0,BTC/ETH,0.896858,-25.252187,0.000000e+00,1.147869,0.002672,0.754293,29
1,BTC/SOL,0.836884,-17.199796,4.616250e-29,1.259260,0.003888,0.726319,37
2,BTC/BNB,0.854269,-27.057060,0.000000e+00,0.795156,0.002284,0.686592,39
3,BTC/XRP,0.859826,-26.766348,0.000000e+00,1.036448,0.002905,0.724760,29
4,BTC/DOGE,0.854656,-27.728218,0.000000e+00,1.099250,0.003152,0.667353,29
...,...,...,...,...,...,...,...,...
185,SUI/ALGO,0.606542,-28.414123,0.000000e+00,0.636800,0.006460,0.634977,32
186,SUI/FIL,0.737592,-15.115379,6.393756e-27,0.868113,0.006150,0.618992,31
187,NEAR/ALGO,0.533153,-29.239859,0.000000e+00,0.416959,0.006874,0.629257,34
188,NEAR/FIL,0.670805,-28.965886,0.000000e+00,0.588107,0.006755,0.616607,26


In [15]:
final_df = final_df.rename(columns={"P-value":"P_value"})

In [16]:
result = final_df.query('Corr > 0.8')
result = result.query('P_value < 0.05')

In [17]:
result = result.sort_values(by="Half-life")
result = result.sort_values(by="Crossings", ascending=False)

In [18]:
selected_pairs = result.reset_index().drop(columns=["index"])

In [20]:
selected_pairs.head()

,Pair,Corr,Score,P_value,Beta,Spread Vol,Half-life,Crossings
0,ETH/LTC,0.803224,-17.490624,3.112398e-29,0.754419,0.003380,0.683868,43
1,DOGE/DOT,0.838430,-27.795916,0.000000e+00,1.011185,0.003991,0.660584,40
2,BTC/BNB,0.854269,-27.057060,0.000000e+00,0.795156,0.002284,0.686592,39
3,XRP/DOT,0.807203,-26.609463,0.000000e+00,1.038768,0.004323,0.655387,39
4,BNB/XRP,0.801326,-26.482018,0.000000e+00,1.037730,0.003404,0.733163,38


In [21]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt

In [22]:
def align_pair_prices(prices, asset_a, asset_b):
    """
    prices: wide DataFrame with columns ['timestamp', asset_a, asset_b, ...]
    returns: DataFrame with only the two assets aligned and sorted
    """
    out = prices[["timestamp", asset_a, asset_b]].dropna().copy()
    out = out.sort_values("timestamp").reset_index(drop=True)
    return out

In [23]:
def estimate_ols_spread(df, asset_a, asset_b):
    """
    Regress log(asset_a) on log(asset_b).
    Returns:
        df_out with log prices, spread, residuals
        alpha, beta, corr, pvalue
    """
    out = df.copy()

    out[f"log_{asset_a}"] = np.log(out[asset_a])
    out[f"log_{asset_b}"] = np.log(out[asset_b])

    X = sm.add_constant(out[f"log_{asset_b}"])
    y = out[f"log_{asset_a}"]

    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params[f"log_{asset_b}"]
    corr = out[[f"log_{asset_a}", f"log_{asset_b}"]].corr().iloc[0, 1]

    out["spread"] = model.resid
    return out, alpha, beta, corr, model.pvalues[f"log_{asset_b}"]

In [24]:
def spread_half_life(spread):
    """
    Estimate half-life from AR(1)-style mean reversion.
    """
    s = pd.Series(spread).dropna()
    lag = s.shift(1)
    diff = s - lag

    tmp = pd.concat([diff, lag], axis=1).dropna()
    tmp.columns = ["diff", "lag"]

    if len(tmp) < 5:
        return np.nan

    model = sm.OLS(tmp["diff"], sm.add_constant(tmp["lag"])).fit()
    lam = model.params["lag"]

    if lam >= 0:
        return np.nan

    return -np.log(2) / lam

In [25]:
def spread_crossings(z, threshold=2.0):
    """
    Count crossings of +/- threshold using one full-sample z-score series.
    """
    z = pd.Series(z).dropna()
    if len(z) < 2:
        return 0

    upper = ((z.shift(1) < threshold) & (z >= threshold)).sum()
    lower = ((z.shift(1) > -threshold) & (z <= -threshold)).sum()
    return int(upper + lower)

In [26]:
def pair_summary(prices, asset_a, asset_b, crossing_z=2.0):
    """
    One-row summary for pair selection.
    """
    df, alpha, beta, corr, pvalue = estimate_ols_spread(prices, asset_a, asset_b)

    spread = df["spread"].dropna()
    spread_vol = spread.std()
    hl = spread_half_life(spread)

    z = (spread - spread.mean()) / spread.std()
    crossings = spread_crossings(z, threshold=crossing_z)

    return pd.Series({
        "Pair": f"{asset_a}/{asset_b}",
        "Asset_A": asset_a,
        "Asset_B": asset_b,
        "Corr": corr,
        "P-value": pvalue,
        "Alpha": alpha,
        "Beta": beta,
        "Spread Vol": spread_vol,
        "Half-life": hl,
        "Z-score Crossings": crossings
    })

In [27]:
def rolling_zscore(series, window=25):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std

In [28]:
def build_zscore_positions(z, entry_z=2.0, exit_z=0.25):
    """
    Stateful position series:
      +1 = long spread
      -1 = short spread
       0 = flat
    """
    position = 0
    positions = []

    for val in z:
        if pd.isna(val):
            positions.append(0)
            continue

        if position == 0:
            if val < -entry_z:
                position = 1
            elif val > entry_z:
                position = -1

        elif position == 1:
            if val > -exit_z:
                position = 0

        elif position == -1:
            if val < exit_z:
                position = 0

        positions.append(position)

    return pd.Series(positions, index=z.index)

In [29]:
def build_bollinger_positions(spread, window=25, band_k=1.0):
    """
    Stateful Bollinger-style spread strategy on the same spread.
    Enter on band break, exit when spread returns to rolling mean.
    """
    mid = spread.rolling(window).mean()
    std = spread.rolling(window).std()
    upper = mid + band_k * std
    lower = mid - band_k * std

    position = 0
    positions = []

    for s, m, u, l in zip(spread, mid, upper, lower):
        if pd.isna(s) or pd.isna(m) or pd.isna(u) or pd.isna(l):
            positions.append(0)
            continue

        if position == 0:
            if s < l:
                position = 1
            elif s > u:
                position = -1

        elif position == 1:
            if s >= m:
                position = 0

        elif position == -1:
            if s <= m:
                position = 0

        positions.append(position)

    out = pd.Series(positions, index=spread.index)
    return out, mid, upper, lower

In [30]:
def backtest_pair_strategy(
    prices,
    asset_a,
    asset_b,
    strategy="zscore",
    beta=None,
    window=25,
    entry_z=2.0,
    exit_z=0.25,
    band_k=1.0,
    fee_bps=0.0
):
    """
    prices: wide DataFrame with timestamp, asset_a, asset_b
    strategy: 'zscore' or 'bollinger'

    Returns:
        bt_df: full backtest DataFrame
        metrics: one-row summary Series
    """
    df = align_pair_prices(prices, asset_a, asset_b)
    df, alpha_hat, beta_hat, corr, pvalue = estimate_ols_spread(df, asset_a, asset_b)

    if beta is None:
        beta = beta_hat

    # Core spread
    df["spread"] = df["spread"]
    df["spread_mean"] = df["spread"].rolling(window).mean()
    df["spread_std"] = df["spread"].rolling(window).std()

    if strategy == "zscore":
        df["zscore"] = rolling_zscore(df["spread"], window=window)
        df["position"] = build_zscore_positions(
            df["zscore"], entry_z=entry_z, exit_z=exit_z
        )

    elif strategy == "bollinger":
        df["position"], df["bb_mid"], df["bb_upper"], df["bb_lower"] = build_bollinger_positions(
            df["spread"], window=window, band_k=band_k
        )

    else:
        raise ValueError("strategy must be 'zscore' or 'bollinger'")

    # Legs
    # position = +1 long spread => long A, short beta * B
    # position = -1 short spread => short A, long beta * B
    df["w_a"] = df["position"].astype(float)
    df["w_b"] = -beta * df["position"].astype(float)

    # Returns
    df[f"ret_{asset_a}"] = df[asset_a].pct_change()
    df[f"ret_{asset_b}"] = df[asset_b].pct_change()

    # Strategy return, shifted to avoid look-ahead
    raw_ret = (
        df["w_a"].shift(1) * df[f"ret_{asset_a}"] +
        df["w_b"].shift(1) * df[f"ret_{asset_b}"]
    )

    # Optional transaction costs on turnover
    turnover = (
        df["w_a"].diff().abs().fillna(0) +
        df["w_b"].diff().abs().fillna(0)
    )

    cost_per_turn = fee_bps / 10000.0
    costs = turnover * cost_per_turn

    df["strategy_ret"] = raw_ret - costs
    df["equity"] = (1 + df["strategy_ret"].fillna(0)).cumprod()

    # Metrics
    r = df["strategy_ret"].dropna()
    ann_factor = np.sqrt(24 * 365)

    sharpe = (r.mean() / r.std()) * ann_factor if r.std() != 0 else np.nan
    max_dd = (df["equity"] / df["equity"].cummax() - 1).min()
    cum_ret = df["equity"].iloc[-1] - 1
    trades = (df["position"].diff().fillna(0) != 0).sum()

    metrics = pd.Series({
        "Pair": f"{asset_a}/{asset_b}",
        "Strategy": strategy,
        "Corr": corr,
        "P-value": pvalue,
        "Alpha": alpha_hat,
        "Beta": beta,
        "Sharpe": sharpe,
        "Cumulative Return": cum_ret,
        "Max Drawdown": max_dd,
        "Trades": int(trades),
        "Mean Hourly Return": r.mean(),
        "Vol Hourly": r.std()
    })

    return df, metrics

In [ ]:
# output selected pairs

In [ ]:
all_results = []
all_backtests = {}

for _, row in selected_pairs.iterrows():
    a = row["Asset_A"]
    b = row["Asset_B"]

    bt_z, m_z = backtest_pair_strategy(
        prices=prices,
        asset_a=a,
        asset_b=b,
        strategy="zscore",
        beta=row.get("Beta", None),
        window=25,
        entry_z=2.0,
        exit_z=0.25,
        fee_bps=0.0
    )

    bt_bb, m_bb = backtest_pair_strategy(
        prices=prices,
        asset_a=a,
        asset_b=b,
        strategy="bollinger",
        beta=row.get("Beta", None),
        window=25,
        band_k=1.0,
        fee_bps=0.0
    )

    all_backtests[(a, b, "zscore")] = bt_z
    all_backtests[(a, b, "bollinger")] = bt_bb

    all_results.append(m_z)
    all_results.append(m_bb)

results_df = pd.DataFrame(all_results).sort_values(["Pair", "Strategy"])
results_df

In [ ]:
candidate_rows = []

for a, b in candidate_pairs:
    candidate_rows.append(pair_summary(prices, a, b))

pair_table = pd.DataFrame(candidate_rows).sort_values("P-value")
pair_table